In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import wandb
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1]
])

dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [5]:
#Generator
class Generator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)
#Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [6]:
#DCGAN
class DCGenerator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 128, 7, 1, 0),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)
#Discriminator
class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128*7*7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [7]:
#Loss Functions
bce = nn.BCELoss()

def d_loss_bce(real, fake):
    real_loss = bce(real, torch.ones_like(real))
    fake_loss = bce(fake, torch.zeros_like(fake))
    return real_loss + fake_loss

def g_loss_bce(fake):
    return bce(fake, torch.ones_like(fake))


# LSGAN
def d_loss_lsgan(real, fake):
    return torch.mean((real - 1)**2 + fake**2)

def g_loss_lsgan(fake):
    return torch.mean((fake - 1)**2)


# WGAN
def d_loss_wgan(real, fake):
    return -(torch.mean(real) - torch.mean(fake))

def g_loss_wgan(fake):
    return -torch.mean(fake)

In [8]:
#Optimizer Selector
def get_optimizer(name, model):
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=0.01)
    elif name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=0.0002)
    else:
        return optim.Adam(model.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [9]:
#Training Loop
def train_gan(model_type="vanilla", loss_type="bce", opt_name="adam", z_dim=100, epochs=5):
    
    wandb.init(project="gan-fashion-mnist", config={
        "model": model_type,
        "loss": loss_type,
        "optimizer": opt_name
    })

    if model_type == "vanilla":
        G = Generator(z_dim).to(device)
        D = Discriminator().to(device)
    else:
        G = DCGenerator(z_dim).to(device)
        D = DCDiscriminator().to(device)

    opt_G = get_optimizer(opt_name, G)
    opt_D = get_optimizer(opt_name, D)

    for epoch in range(epochs):
        for i, (real, _) in enumerate(loader):
            real = real.to(device)
            batch_size = real.size(0)

            # Noise
            if model_type == "vanilla":
                z = torch.randn(batch_size, z_dim).to(device)
            else:
                z = torch.randn(batch_size, z_dim, 1, 1).to(device)

            fake = G(z)

            # ---- Train Discriminator ----
            D_real = D(real)
            D_fake = D(fake.detach())

            if loss_type == "bce":
                loss_D = d_loss_bce(D_real, D_fake)
            elif loss_type == "lsgan":
                loss_D = d_loss_lsgan(D_real, D_fake)
            else:
                loss_D = d_loss_wgan(D_real, D_fake)

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # ---- Train Generator ----
            D_fake = D(fake)

            if loss_type == "bce":
                loss_G = g_loss_bce(D_fake)
            elif loss_type == "lsgan":
                loss_G = g_loss_lsgan(D_fake)
            else:
                loss_G = g_loss_wgan(D_fake)

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        print(f"Epoch {epoch} | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")

        wandb.log({
            "D_loss": loss_D.item(),
            "G_loss": loss_G.item()
        })

        # Log images
        with torch.no_grad():
            sample = G(z)[:5]
            wandb.log({"generated_images": [wandb.Image(img) for img in sample]})

In [ ]:
models = ["vanilla", "dcgan"]
losses = ["bce", "lsgan", "wgan"]
opts = ["adam"]

for m in models:
    for l in losses:
        for o in opts:
            train_gan(m, l, o)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jahanavisingh/.netrc.
wandb: Currently logged in as: jahanavisingh_25rco12 (jahanavisingh_25rco12-delhi-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.


Epoch 0 | D Loss: 0.0796 | G Loss: 5.1544
Epoch 1 | D Loss: 0.3941 | G Loss: 1.8881
Epoch 2 | D Loss: 0.3265 | G Loss: 3.1081
Epoch 3 | D Loss: 0.4911 | G Loss: 2.4998
Epoch 4 | D Loss: 0.7052 | G Loss: 1.5361


D_loss,▁▅▄▆█
G_loss,█▂▄▃▁
D_loss,0.70523
G_loss,1.53615


Epoch 0 | D Loss: 0.2333 | G Loss: 0.9719
Epoch 1 | D Loss: 0.2096 | G Loss: 0.7665
Epoch 2 | D Loss: 0.2113 | G Loss: 0.8018
Epoch 3 | D Loss: 0.3329 | G Loss: 0.9113
Epoch 4 | D Loss: 0.2100 | G Loss: 0.6510


D_loss,▂▁▁█▁
G_loss,█▄▄▇▁
D_loss,0.21001
G_loss,0.651


Epoch 0 | D Loss: -0.0000 | G Loss: -1.0000
Epoch 1 | D Loss: 0.0000 | G Loss: -1.0000
Epoch 2 | D Loss: -0.0000 | G Loss: -1.0000
Epoch 3 | D Loss: -0.0000 | G Loss: -1.0000
Epoch 4 | D Loss: -0.0000 | G Loss: -1.0000


D_loss,▁█▂▃▄
G_loss,█▃▅▃▁
D_loss,-0.0
G_loss,-0.99999


Epoch 0 | D Loss: 0.9897 | G Loss: 2.8865
Epoch 1 | D Loss: 0.9058 | G Loss: 2.1489
Epoch 2 | D Loss: 1.0268 | G Loss: 0.7693
Epoch 3 | D Loss: 0.9667 | G Loss: 0.9878
Epoch 4 | D Loss: 1.0522 | G Loss: 0.9327
